# 04 - Mapping Strategy

## Purpose

- The purpose of this notebook is to define and validate the mapping strategy between the source datasets and the target Abicart import format.
- This includes identifying how fields from the Price List, Product Feed, and existing Abicart export correspond to the final import schema. The notebook documents all mapping decisions, highlights ambiguities and records assumptions that must be validated before any transformations are implemented.
- No data transformations are performed in this notebook. Its sole purpose is to establish a reproducible and well-documented mapping specification for subsequent notebooks.

## Imports

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

## File Configuration

In [ ]:
SUPPLIER = "snickers"


PRICE_LIST_PATH = (
    f"../data/{SUPPLIER}/price_list/Prislista_Snickers_WW_202609.xlsx"
)

PRODUCT_FEED_PATH = (
    f"../data/{SUPPLIER}/product_feeds/PP_Export_Snickers_018_sv.xml"
)

ABICART_EXPORT_PATH = (
    f"../data/{SUPPLIER}/abicart/abicart_full_export.csv"
)

ABICART_IMPORT_SAMPLE_PATH = (
    f"../data/{SUPPLIER}/abicart/import_exempelfil.csv"
)

## Load Datasets

In [ ]:
price_list = pd.read_excel(
    PRICE_LIST_PATH,
    header=1,
)

tree = ET.parse(PRODUCT_FEED_PATH)
root = tree.getroot()

abicart_df = pd.read_csv(
    ABICART_EXPORT_PATH,
    encoding="latin1",
    skiprows=1,
    low_memory=False,
)

abicart_import_sample = pd.read_csv(
    ABICART_IMPORT_SAMPLE_PATH,
    encoding="utf-8-sig",
)

## Mapping Strategy

This notebook defines the transformation from supplier source data to the Abicart import structure.

## Principles:
- The current supplier Price List defines the active assortment and is the source of truth for variants and current commercial data.
- Product Feed enriches the active assortment with product content and media.
- Product level = supplier model.
- Variant level = supplier article number.
- Customer choices are standardized to `Färg` and `Storlek`.
- `Produktnummer under kundens val` uses the supplier's exact variant article number.
- Historical manually constructed Abicart variant identifiers are not used as the standard for new imports.
- Existing Abicart data represents the current target state, not the source of truth for supplier product data.
- Source analysis belongs in notebooks 01–03; this notebook defines mapping and transformation logic.

### Source Roles

| Source | Role |
|---|---|
| Price List | Driving dataset and source of truth for the current supplier assortment, variants, article numbers, prices, EAN, colours and sizes. |
| Product Feed | Enrichment source for product content such as descriptions, features and images. |
| Abicart Full Export | Existing target state used for matching products and determining later update/replace/hide/leave-unchanged actions. |
| Abicart Import Sample | Defines the standard Abicart product fields. Customer-choice fields are added based on the verified Abicart export structure and import mapping. |

## Mapping Analysis

### Product Identifier Mapping

| Level | Source | Source field | Abicart target | Transformation |
|---|---|---|---|---|
| Product | Price List | `Modell` | `Article number (required)` | Use the supplier model number as the parent product identifier. |
| Variant | Price List | `Artikelnr` | `Produktnummer under kundens val` | Use the supplier's exact variant article number without constructing a new identifier. |

### Product and Variant Structure

Each supplier model becomes one parent product in Abicart.

Variants are created from the supplier's variant-level records:

- Parent product: `Modell`
- Customer choice 1: `Färg`
- Customer choice 2: `Storlek`
- Variant identifier: exact supplier `ArtikelNr`
- `Produktnummer under kundens val`: exact supplier `ArtikelNr`

No variant article numbers are constructed manually.

### Product Name Mapping

| Level | Source | Source field | Abicart target | Transformation |
|---|---|---|---|---|
| Product | Product Feed | `Name` | `Name` | Use the Product Feed name for the parent product. |
| Variant | — | — | — | Variants inherit the product name; no separate variant product name is generated. |

### Description Mapping

| Level | Source | Source field | Abicart target | Transformation |
|---|---|---|---|---|
| Product | Product Feed | `Intro` | `Description` | Use the supplier product intro as the parent product description. |
| Variant | — | — | — | Description is stored at product level and is not generated separately for each variant. |

### Price Mapping

| Level | Source | Source field | Abicart target | Transformation |
|---|---|---|---|---|
| Variant | Price List | `RRP Pris` | `Price` | Use the current supplier RRP from the active Price List. Variants identified as special-priced variants (based on the supplier pricing within the same product model) are excluded from the webshop import. |

### EAN Analysis
The purpose of this section is to determine how EAN information is represented across the source datasets and how it should be mapped to the target Abicart import structure. 

In [ ]:
selected_columns = abicart_df.columns[8:14]

print(selected_columns.tolist())
display(abicart_df.loc[:, selected_columns].head(10))

Observed:
- The Price List contains EAN values in `EAN-nr. styck`.
- The Product Feed contains EAN values in `EAN_text`.
- The existing Abicart export contains partial historical EAN coverage in `EAN-kod`.
- Whether EAN should be included in the generated import remains an implementation decision.

### Image Analysis
The purpose of this section is to determine how product images are represented across the source datasets and how they should be mapped to the target Abicart import structure.

Observed:
- No image-related fields were identified in the Price List structure.

Observed: 
- The Product Feed contains a dedicated main image field with an image URL:
- In the inspected Abicart export, image URLs are stored in column 8 on parent product rows.

| Target Field | Price List | Product Feed | Abicart | Notes |
|---|---|---|---|---|
| Product Identifier | `ArtikelNr` | `StockCode` | Parent ID / Variant ID | Price List `ArtikelNr` is the current supplier variant identifier. |
| Product Name | `Beskrivning 1` | `Name` | Column 4 | Product Feed provides the parent product name. |
| Description | — | `Intro` | Column 5 | Description is stored at parent product level. |
| Price | `RRP Pris` | — | Column 6 | Price List is the source of truth for current pricing. |
| Image | — | Main image field | Column 8 | Product Feed provides the source image URL. |

## Customer Choice Mapping

### Purpose

Customer choices are used to represent supplier product variants in a consistent and reusable way for Abicart imports.

### Mapping

| Customer Choice | Source | Source Field | Target | Notes |
|---|---|---|---|---|
| Parent Product | Price List | `Modell` | Parent product | One parent product is created per supplier model. |
| Colour | Price List | `Färg` | Customer Choice: `Färg` | One customer choice value per colour. |
| Size | Price List | `Storlekskod` | Customer Choice: `Storlek` | Standardised customer choice name in Abicart. |
| Variant Identifier | Price List | `Artikelnr` | `Produktnummer under kundens val` | Preserve the supplier article number without modification. |

### Design Decisions

- The supplier model (`Modell`) is used as the parent product identifier.
- Each unique combination of `Färg` and `Storlekskod` represents one product variant.
- The supplier article number (`Artikelnr`) is preserved as the variant identifier.
- Customer choice names are standardised to `Färg` and `Storlek` for all future imports.
- No manually generated variant article numbers will be used.
- Supplier variants identified as special-priced variants within the same product model are excluded from the webshop import.

## Transformation Workflow

The transformation process converts the supplier source datasets into a valid Abicart import file by combining product data from the current supplier Price List, Product Feed and the existing Abicart export.

The Workflow consists of the following high-level steps:

1. Load datasets.
2. Validate required identifiers.
3. Filter active assortment.
4. Remove supplier special-priced variants.
5. Merge Price List with Product Feed.
6. Apply field mapping.
7. Create parent products.
8. Create product variants.
9. Export Abicart import file.

## Findings

- Product identification exists at both model and variant level across the source datasets.
- The Price List serves as the primary source for variant-level pricing and EAN information.
- The Product Feed serves as the primary source for descriptions, images and marketing content.
- The Abicart export uses a parent/variant structure.
- EAN information is available in both supplier sources and has partial historical coverage in the existing Abicart export.
- Supplier special-order variants are excluded before the final Abicart import is generated.

## Open Questions

- Should EAN values be included in future webshop imports?
- Are additional Product Feed fields (e.g., sustainability and certification fields) relevant for webshop presentation?
- Are additional category mappings required to support webshop navigation?